# 04 — Análise Espectral 100 Hz | 7 Classes

Pipeline completo de análise espectral para calibração de bandas e treino do modelo GNB de 7 classes.

**Coleta**: 100 Hz, janela 1000 amostras (10 s) → resolução Δf = 0.1 Hz  
**Pipeline FFT**: detrend linear → janela Hann → zero-padding n_fft=4096 → bins a 0.024 Hz  
**Classes**: FAN_OFF · LOW_ROT_OFF · LOW_ROT_ON · MEDIUM_ROT_OFF · MEDIUM_ROT_ON · HIGH_ROT_OFF · HIGH_ROT_ON

In [23]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'shared'))

import numpy as np
import pandas as pd
import json
import time
import oracledb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.stats import f_oneway

from feature_engineering import (
    compute_spectral_signature,
    compute_spectral_features,
    compute_time_features,
    extract_features_windowed_spectral,
    extract_features_windowed,
    SPECTRAL_BANDS,
)

# ── Parâmetros da coleta ──────────────────────────────────────────────────────
COLLECTION_ID = "col_20260223_115149_100hz"
SAMPLE_HZ     = 100
WINDOW_SIZE   = 1000    # 10 s → Δf = 0.1 Hz
STEP_SIZE     = 500     # 50 % overlap → nova análise a cada 5 s
N_FFT         = 4096    # zero-padding → bins a 0.024 Hz

AXES = ['accel_x_g', 'accel_y_g', 'accel_z_g',
        'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps']

CLASSES_ORDER = [
    'FAN_OFF',
    'LOW_ROT_OFF',  'LOW_ROT_ON',
    'MEDIUM_ROT_OFF','MEDIUM_ROT_ON',
    'HIGH_ROT_OFF', 'HIGH_ROT_ON',
]

COMPOSITE_MAP = {
    ('OFF',    'STOPPED'):  'FAN_OFF',
    ('LOW',    'STOPPED'):  'LOW_ROT_OFF',
    ('LOW',    'ROTATING'): 'LOW_ROT_ON',
    ('MEDIUM', 'STOPPED'):  'MEDIUM_ROT_OFF',
    ('MEDIUM', 'ROTATING'): 'MEDIUM_ROT_ON',
    ('HIGH',   'STOPPED'):  'HIGH_ROT_OFF',
    ('HIGH',   'ROTATING'): 'HIGH_ROT_ON',
}

COLORS = {
    'FAN_OFF':        '#888888',
    'LOW_ROT_OFF':    '#1f77b4',
    'LOW_ROT_ON':     '#aec7e8',
    'MEDIUM_ROT_OFF': '#ff7f0e',
    'MEDIUM_ROT_ON':  '#ffbb78',
    'HIGH_ROT_OFF':   '#d62728',
    'HIGH_ROT_ON':    '#ff9896',
}

print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"SAMPLE_HZ     : {SAMPLE_HZ} Hz  |  Nyquist: {SAMPLE_HZ/2} Hz")
print(f"WINDOW_SIZE   : {WINDOW_SIZE} pts = {WINDOW_SIZE/SAMPLE_HZ:.0f} s")
print(f"N_FFT         : {N_FFT}  →  bin spacing: {SAMPLE_HZ/N_FFT:.4f} Hz")

COLLECTION_ID : col_20260223_115149_100hz
SAMPLE_HZ     : 100 Hz  |  Nyquist: 50.0 Hz
WINDOW_SIZE   : 1000 pts = 10 s
N_FFT         : 4096  →  bin spacing: 0.0244 Hz


## 1. Carga de Dados — Oracle (transition_marker = 0)

In [24]:
conn = oracledb.connect(user="dersao", password="986960440", dsn="localhost:1521/xepdb1")

sql = """
    SELECT ts_epoch, accel_x_g, accel_y_g, accel_z_g,
           gyro_x_dps, gyro_y_dps, gyro_z_dps,
           cmd_speed_label, rot_state_label, transition_marker, sample_rate
    FROM sensor_data
    WHERE collection_id    = :cid
      AND transition_marker = 0
    ORDER BY ts_epoch ASC
"""
df_raw = pd.read_sql(sql, conn, params={"cid": COLLECTION_ID})
conn.close()

# Normalizar nomes de colunas (Oracle retorna em maiúsculas)
df_raw.columns = [c.lower() for c in df_raw.columns]

# Reconstruir label composto
df_raw['fan_state'] = df_raw.apply(
    lambda r: COMPOSITE_MAP.get(
        (str(r['cmd_speed_label']).upper(), str(r['rot_state_label']).upper()),
        'UNKNOWN'
    ), axis=1
)

# ts_epoch é epoch em segundos (NUMBER 18,6)
df_raw['timestamp_s'] = df_raw['ts_epoch'].astype(float)

print(f"Total amostras (transition=0): {len(df_raw):,}")
print(f"\nAmostras por classe:")
vc = df_raw['fan_state'].value_counts()
for cls in CLASSES_ORDER:
    n = vc.get(cls, 0)
    dur = n / SAMPLE_HZ
    print(f"  {cls:<20} {n:>7,} amostras  ({dur:.0f} s)")

C:\Users\Anderson\AppData\Local\Temp\ipykernel_6128\391712853.py:12: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



Total amostras (transition=0): 130,060

Amostras por classe:
  FAN_OFF               17,680 amostras  (177 s)
  LOW_ROT_OFF           17,440 amostras  (174 s)
  LOW_ROT_ON            19,600 amostras  (196 s)
  MEDIUM_ROT_OFF        18,240 amostras  (182 s)
  MEDIUM_ROT_ON         17,960 amostras  (180 s)
  HIGH_ROT_OFF          18,040 amostras  (180 s)
  HIGH_ROT_ON           21,100 amostras  (211 s)


## 2. Assinaturas Espectrais (FFT médio por classe)

Cada curva = média de N janelas de 10 s.  
Picos dominantes identificam a frequência característica de cada classe.  
**Use este gráfico para calibrar os SPECTRAL_BANDS.**

In [25]:
def mean_spectrum(df_class, axis, n_windows=20):
    """Média das assinaturas espectrais de n_windows janelas aleatórias."""
    n = len(df_class)
    if n < WINDOW_SIZE:
        return None, None, None, None
    starts = np.linspace(0, n - WINDOW_SIZE, min(n_windows, (n - WINDOW_SIZE) // STEP_SIZE + 1), dtype=int)
    all_mags = []
    for s in starts:
        vals = df_class[axis].iloc[s:s + WINDOW_SIZE].values
        freqs, mags, pf, pm, _ = compute_spectral_signature(vals, sampling_hz=SAMPLE_HZ, n_fft=N_FFT)
        all_mags.append(mags)
    avg = np.mean(all_mags, axis=0)
    return freqs, avg

# ── Plot: 1 figura por eixo, todas as 7 classes sobrepostas ──────────────────
os.makedirs('output/figures', exist_ok=True)

for axis in AXES:
    fig = go.Figure()
    peak_table = []

    for cls in CLASSES_ORDER:
        df_cls = df_raw[df_raw['fan_state'] == cls].reset_index(drop=True)
        freqs, avg_mags = mean_spectrum(df_cls, axis)
        if freqs is None:
            continue

        # Limitar display a 0–30 Hz (acima do noise floor é irrelevante)
        mask = freqs <= 30
        f_show = freqs[mask]
        m_show = avg_mags[mask]

        # Pico dominante acima de 0.5 Hz
        mask_peak = f_show >= 0.5
        if mask_peak.any():
            idx_pk = int(np.argmax(m_show[mask_peak]))
            peak_f = float(f_show[mask_peak][idx_pk])
            peak_m = float(m_show[mask_peak][idx_pk])
        else:
            peak_f, peak_m = 0.0, 0.0

        peak_table.append({'Classe': cls, 'Eixo': axis,
                           'Pico Hz': round(peak_f, 3),
                           'Magnitude': round(peak_m, 6)})

        fig.add_trace(go.Scatter(
            x=f_show, y=m_show,
            name=cls, line=dict(color=COLORS[cls], width=1.5),
            hovertemplate=f'<b>{cls}</b><br>f=%{{x:.3f}} Hz<br>mag=%{{y:.6f}}<extra></extra>',
        ))

    # Linhas verticais das bandas atuais
    for band, (flo, fhi) in SPECTRAL_BANDS.items():
        fig.add_vrect(x0=flo, x1=fhi, fillcolor='rgba(200,200,200,0.15)',
                      line_width=0, annotation_text=band, annotation_position='top left')

    fig.update_layout(
        title=f'Assinaturas Espectrais — {axis}  (100 Hz, Hann + zero-pad)',
        xaxis_title='Frequência (Hz)',
        yaxis_title='Magnitude (corrigida)',
        hovermode='x unified',
        height=420,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig.write_html(f'output/figures/04_spectrum_{axis}.html')
    fig.show()

    df_peaks = pd.DataFrame(peak_table)
    print(f"\n{'─'*55}")
    print(f"  {axis} — picos dominantes:")
    print(df_peaks[['Classe', 'Pico Hz', 'Magnitude']].to_string(index=False))


───────────────────────────────────────────────────────
  accel_x_g — picos dominantes:
        Classe  Pico Hz  Magnitude
       FAN_OFF    2.197   0.000558
   LOW_ROT_OFF    1.465   0.016341
    LOW_ROT_ON    1.025   0.016227
MEDIUM_ROT_OFF    2.197   0.010331
 MEDIUM_ROT_ON    2.246   0.025797
  HIGH_ROT_OFF    5.322   0.042614
   HIGH_ROT_ON    3.931   0.042638



───────────────────────────────────────────────────────
  accel_y_g — picos dominantes:
        Classe  Pico Hz  Magnitude
       FAN_OFF    1.050   0.000672
   LOW_ROT_OFF    1.465   0.012797
    LOW_ROT_ON    1.001   0.014867
MEDIUM_ROT_OFF    2.173   0.016949
 MEDIUM_ROT_ON    1.929   0.020191
  HIGH_ROT_OFF    2.344   0.025187
   HIGH_ROT_ON    3.418   0.037995



───────────────────────────────────────────────────────
  accel_z_g — picos dominantes:
        Classe  Pico Hz  Magnitude
       FAN_OFF    5.371   0.000393
   LOW_ROT_OFF    1.465   0.005006
    LOW_ROT_ON   19.434   0.003489
MEDIUM_ROT_OFF    2.173   0.004964
 MEDIUM_ROT_ON   20.117   0.006604
  HIGH_ROT_OFF   20.679   0.007071
   HIGH_ROT_ON   20.068   0.010182



───────────────────────────────────────────────────────
  gyro_x_dps — picos dominantes:
        Classe  Pico Hz  Magnitude
       FAN_OFF    1.782   0.027566
   LOW_ROT_OFF    1.440   0.379235
    LOW_ROT_ON    1.025   0.425802
MEDIUM_ROT_OFF   23.120   0.434019
 MEDIUM_ROT_ON   21.216   0.696437
  HIGH_ROT_OFF    3.369   0.970342
   HIGH_ROT_ON    3.418   1.342101



───────────────────────────────────────────────────────
  gyro_y_dps — picos dominantes:
        Classe  Pico Hz  Magnitude
       FAN_OFF    2.100   0.028571
   LOW_ROT_OFF    1.465   0.331983
    LOW_ROT_ON    0.903   0.207509
MEDIUM_ROT_OFF   27.002   0.219359
 MEDIUM_ROT_ON   12.061   0.403308
  HIGH_ROT_OFF    9.302   0.469994
   HIGH_ROT_ON    9.814   0.518655



───────────────────────────────────────────────────────
  gyro_z_dps — picos dominantes:
        Classe  Pico Hz  Magnitude
       FAN_OFF    1.807   0.026743
   LOW_ROT_OFF    1.465   0.776160
    LOW_ROT_ON    1.074   2.541185
MEDIUM_ROT_OFF    2.197   2.122147
 MEDIUM_ROT_ON    1.953   3.287969
  HIGH_ROT_OFF    2.368   1.947393
   HIGH_ROT_ON    3.931   2.141754


## 3. Calibração dos SPECTRAL_BANDS

Ajuste os limites abaixo com base nos picos observados no gráfico acima.  
Execute a célula para atualizar `feature_engineering.py` automaticamente.

In [26]:
# ── EDITE AQUI após olhar os gráficos ────────────────────────────────────────
# Substitua pelos picos reais observados (±50 % de margem ao redor do pico)
BANDS_CALIBRATED = {
    'rot':   (0.10,  1.50),   # rotação do suporte (~0.25 Hz = 1 rot/4s) — AJUSTAR
    'vel1':  (3.00,  6.50),   # LOW  speed (~5 Hz) — AJUSTAR
    'vel2':  (6.50,  9.50),   # MEDIUM speed (~7.5 Hz) — AJUSTAR
    'vel3':  (9.50, 14.00),   # HIGH speed (~10–12 Hz) — AJUSTAR
    'noise': (25.0,  50.0),   # noise floor (referência)
}

print("Bandas atuais vs calibradas:")
for name in BANDS_CALIBRATED:
    old = SPECTRAL_BANDS.get(name, ('?','?'))
    new = BANDS_CALIBRATED[name]
    changed = "← ALTERADO" if old != new else ""
    print(f"  {name:<8}  atual={old}  novo={new}  {changed}")

# ── Confirmar antes de salvar ─────────────────────────────────────────────────
# Descomente a linha abaixo APÓS confirmar os valores com os gráficos:
# _SAVE_BANDS = True
_SAVE_BANDS = False  # mude para True quando os valores estiverem corretos

if _SAVE_BANDS:
    fe_path = os.path.join('shared', 'feature_engineering.py')
    with open(fe_path, 'r', encoding='utf-8') as f:
        src = f.read()

    import re
    new_bands_str = "SPECTRAL_BANDS = {\n"
    for name, (flo, fhi) in BANDS_CALIBRATED.items():
        new_bands_str += f"    '{name}': ({flo:.2f}, {fhi:.2f}),\n"
    new_bands_str += "}"

    src_new = re.sub(
        r"SPECTRAL_BANDS\s*=\s*\{[^}]*\}",
        new_bands_str,
        src,
        flags=re.DOTALL,
    )
    with open(fe_path, 'w', encoding='utf-8') as f:
        f.write(src_new)
    print("\n✓ feature_engineering.py atualizado com novos SPECTRAL_BANDS")
else:
    print("\n(Edite _SAVE_BANDS = True para persistir)")

Bandas atuais vs calibradas:
  rot       atual=(0.1, 1.5)  novo=(0.1, 1.5)  
  vel1      atual=(3.0, 6.5)  novo=(3.0, 6.5)  
  vel2      atual=(6.5, 9.5)  novo=(6.5, 9.5)  
  vel3      atual=(9.5, 14.0)  novo=(9.5, 14.0)  
  noise     atual=(25.0, 50.0)  novo=(25.0, 50.0)  

(Edite _SAVE_BANDS = True para persistir)


## 4. Extração de Features (Temporal + Espectral)

In [27]:
rows_all = []

for cls in CLASSES_ORDER:
    df_cls = df_raw[df_raw['fan_state'] == cls].reset_index(drop=True)
    n = len(df_cls)
    if n < WINDOW_SIZE:
        print(f"  {cls}: insuficiente ({n} amostras)")
        continue

    # Features espectrais (detrend + Hann + FFT)
    spec_rows = extract_features_windowed_spectral(
        df_cls, cls, AXES, WINDOW_SIZE, STEP_SIZE,
        timestamp_col='timestamp_s', sampling_hz=SAMPLE_HZ, n_fft=N_FFT,
    )
    # Features temporais (11 métricas)
    temp_rows = extract_features_windowed(
        df_cls, cls, AXES, WINDOW_SIZE, STEP_SIZE,
        timestamp_col='timestamp_s',
    )

    # Merge: mesmas janelas, junte features
    for sr, tr in zip(spec_rows, temp_rows):
        merged = {**tr, **{k: v for k, v in sr.items()
                           if k not in ('fan_state','collection_id','window_start',
                                        'window_end','timestamp_start','timestamp_end','timestamp_mean')}}
        rows_all.append(merged)

    print(f"  {cls:<20}  {len(spec_rows):>4} janelas  ({n} amostras)")

df_feat = pd.DataFrame(rows_all)
y = df_feat['fan_state'].values

feature_cols = [c for c in df_feat.columns
                if c not in ('fan_state','collection_id','window_start','window_end',
                             'timestamp_start','timestamp_end','timestamp_mean')]
X = df_feat[feature_cols].fillna(0)

temp_cols    = [c for c in feature_cols if '_band_' not in c and '_peak_freq' not in c and '_peak_mag' not in c]
spectral_cols = [c for c in feature_cols if c not in temp_cols]

print(f"\nDataset: {len(X)} janelas × {len(feature_cols)} features")
print(f"  Temporais : {len(temp_cols)}")
print(f"  Espectrais: {len(spectral_cols)}")
print(f"  Classes   : {pd.Series(y).value_counts().to_dict()}")

  FAN_OFF                 34 janelas  (17680 amostras)
  LOW_ROT_OFF             33 janelas  (17440 amostras)
  LOW_ROT_ON              38 janelas  (19600 amostras)
  MEDIUM_ROT_OFF          35 janelas  (18240 amostras)
  MEDIUM_ROT_ON           34 janelas  (17960 amostras)
  HIGH_ROT_OFF            35 janelas  (18040 amostras)
  HIGH_ROT_ON             41 janelas  (21100 amostras)

Dataset: 250 janelas × 108 features
  Temporais : 66
  Espectrais: 42
  Classes   : {'HIGH_ROT_ON': 41, 'LOW_ROT_ON': 38, 'MEDIUM_ROT_OFF': 35, 'HIGH_ROT_OFF': 35, 'FAN_OFF': 34, 'MEDIUM_ROT_ON': 34, 'LOW_ROT_OFF': 33}


## 5. Ranking ANOVA — Top Features por Separabilidade

In [28]:
anova_rows = []
for col in feature_cols:
    groups = [X[col][y == c].values for c in CLASSES_ORDER if (y == c).sum() > 1]
    if len(groups) >= 2:
        f_stat, p_val = f_oneway(*groups)
        anova_rows.append({
            'feature': col,
            'f_stat': f_stat,
            'p_value': p_val,
            'type': 'spectral' if col in spectral_cols else 'temporal',
        })

df_anova = pd.DataFrame(anova_rows).sort_values('f_stat', ascending=False)
df_sig   = df_anova[df_anova['p_value'] < 0.05]

print(f"Features significativas (p<0.05): {len(df_sig)}/{len(df_anova)}")
print(f"  Temporais : {(df_sig['type']=='temporal').sum()}")
print(f"  Espectrais: {(df_sig['type']=='spectral').sum()}")
print(f"\nTop 25 features:")
print(df_sig.head(25)[['feature','f_stat','p_value','type']].to_string(index=False))

Features significativas (p<0.05): 105/108
  Temporais : 63
  Espectrais: 42

Top 25 features:
                   feature      f_stat       p_value     type
             accel_x_g_std 7096.124521 9.555966e-270 temporal
           gyro_z_dps_peak 6984.264269 6.513570e-269 temporal
    accel_x_g_shape_factor 6114.060342 6.186301e-262 temporal
            accel_x_g_peak 4234.973452 1.027489e-242 temporal
             accel_z_g_std 3970.468030 2.404572e-239 temporal
            gyro_z_dps_rms 3829.853243 1.835684e-237 temporal
            gyro_x_dps_std 3783.460138 7.943522e-237 temporal
            accel_y_g_peak 3459.314099 3.741681e-232 temporal
            gyro_x_dps_rms 3435.636882 8.534795e-232 temporal
            gyro_y_dps_std 3424.939133 1.241083e-231 temporal
            gyro_y_dps_rms 3043.179775 1.783417e-225 temporal
accel_x_g_clearance_factor 3010.481191 6.510939e-225 temporal
 gyro_x_dps_root_amplitude 2899.935880 5.762190e-223 temporal
             accel_y_g_std 2862.056550

## 6. Treino GNB — 7 Classes (CV 5-fold)

In [29]:
def remove_correlated(feat_list, X_df, threshold=0.85):
    """Remove features altamente correlacionadas, mantendo maior F-stat."""
    if len(feat_list) < 2:
        return feat_list
    corr = X_df[feat_list].corr().abs()
    rank = {f: i for i, f in enumerate(df_anova['feature'].values)}
    to_drop = set()
    for i in range(len(feat_list)):
        for j in range(i + 1, len(feat_list)):
            if corr.iloc[i, j] > threshold:
                fi, fj = feat_list[i], feat_list[j]
                drop = fj if rank.get(fi, 9999) < rank.get(fj, 9999) else fi
                to_drop.add(drop)
    return [f for f in feat_list if f not in to_drop]

sig_set = set(df_sig['feature'])

# Conjunto A: melhores temporais
feat_temp = [f for f in df_anova['feature'] if f in sig_set and f in temp_cols]
set_A = remove_correlated(feat_temp[:30], X)

# Conjunto B: melhores espectrais
feat_spec = [f for f in df_anova['feature'] if f in sig_set and f in spectral_cols]
set_B = remove_correlated(feat_spec[:30], X)

# Conjunto C: combinado top-40
feat_comb = [f for f in df_anova['feature'] if f in sig_set][:40]
set_C = remove_correlated(feat_comb, X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, fset in [('A temporal', set_A), ('B spectral', set_B), ('C combined', set_C)]:
    if not fset:
        print(f"{name}: sem features")
        continue
    scores = cross_val_score(GaussianNB(), X[fset], y, cv=cv, scoring='accuracy')
    results[name] = (scores, fset)
    print(f"  {name}:  {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%  ({len(fset)} features)")

best_name = max(results, key=lambda k: results[k][0].mean())
best_features = results[best_name][1]
print(f"\nMelhor conjunto: {best_name}")
print(f"Features selecionadas: {best_features}")

  A temporal:  100.00% ± 0.00%  (3 features)
  B spectral:  89.60% ± 4.96%  (2 features)
  C combined:  100.00% ± 0.00%  (3 features)

Melhor conjunto: A temporal
Features selecionadas: ['accel_x_g_std', 'gyro_z_dps_peak', 'gyro_y_dps_skew']


## 7. Matriz de Confusão — 7 Classes

In [30]:
clf = GaussianNB()
clf.fit(X[best_features], y)
y_pred = clf.predict(X[best_features])

print(classification_report(y, y_pred, labels=CLASSES_ORDER, zero_division=0))

cm = confusion_matrix(y, y_pred, labels=CLASSES_ORDER)
fig = px.imshow(
    cm,
    x=CLASSES_ORDER, y=CLASSES_ORDER,
    text_auto=True, color_continuous_scale='Blues',
    labels=dict(x='Predito', y='Real'),
    title=f'Matriz de Confusão — {best_name} ({len(best_features)} features)',
)
fig.update_layout(height=500)
fig.write_html('output/figures/04_confusion_matrix_7class.html')
fig.show()

                precision    recall  f1-score   support

       FAN_OFF       1.00      1.00      1.00        34
   LOW_ROT_OFF       1.00      1.00      1.00        33
    LOW_ROT_ON       1.00      1.00      1.00        38
MEDIUM_ROT_OFF       1.00      1.00      1.00        35
 MEDIUM_ROT_ON       1.00      1.00      1.00        34
  HIGH_ROT_OFF       1.00      1.00      1.00        35
   HIGH_ROT_ON       1.00      1.00      1.00        41

      accuracy                           1.00       250
     macro avg       1.00      1.00      1.00       250
  weighted avg       1.00      1.00      1.00       250



## 8. Export — Modelo GNB JSON (7 classes, 100 Hz)

In [31]:
cv_scores = cross_val_score(clf, X[best_features], y, cv=cv)
train_acc = accuracy_score(y, y_pred)
ts = time.strftime('%Y%m%d_%H%M%S')

model_data = {
    'type':             'gaussian_nb',
    'version':          f'7class_100hz_{ts}',
    'generated_at':     time.strftime('%Y-%m-%d %H:%M:%S'),
    'sample_rate_hz':   SAMPLE_HZ,
    'window_size':      WINDOW_SIZE,
    'step_size':        STEP_SIZE,
    'n_fft':            N_FFT,
    'collection_id':    COLLECTION_ID,
    'features':         list(best_features),
    'labels':           list(clf.classes_),
    'priors':           {lbl: float(clf.class_prior_[i]) for i, lbl in enumerate(clf.classes_)},
    'stats':            {},
    'metrics': {
        'train_accuracy':   float(train_acc),
        'cv_accuracy_mean': float(cv_scores.mean()),
        'cv_accuracy_std':  float(cv_scores.std()),
    },
    'spectral_bands':   SPECTRAL_BANDS,
}

for i, lbl in enumerate(clf.classes_):
    model_data['stats'][lbl] = {
        feat: {'mean': float(clf.theta_[i, j]), 'var': float(clf.var_[i, j])}
        for j, feat in enumerate(best_features)
    }

date_str = time.strftime('%Y%m%d')
for path in [
    f'output/models/gnb_model_{date_str}.json',
    f'../models/gnb_model_{date_str}.json',
]:
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
    with open(path, 'w') as f:
        json.dump(model_data, f, indent=2)
    print(f"✓ Modelo exportado: {path}")

print(f"\nFeatures ({len(best_features)}): {best_features}")
print(f"Acurácia CV: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
print(f"\nPróximo passo:")
print(f"  1. Olhe os gráficos espectrais e calibre BANDS_CALIBRATED na Célula 3")
print(f"  2. Se SPECTRAL_BANDS mudar, re-execute a partir da Célula 4")
print(f"  3. Use o modelo exportado no backend e no JS do frontend")

✓ Modelo exportado: output/models/gnb_model_20260223.json
✓ Modelo exportado: ../models/gnb_model_20260223.json

Features (3): ['accel_x_g_std', 'gyro_z_dps_peak', 'gyro_y_dps_skew']
Acurácia CV: 100.00% ± 0.00%

Próximo passo:
  1. Olhe os gráficos espectrais e calibre BANDS_CALIBRATED na Célula 3
  2. Se SPECTRAL_BANDS mudar, re-execute a partir da Célula 4
  3. Use o modelo exportado no backend e no JS do frontend
